# Part 15 (실습) — 멀티에이전트: 감독자와 전문가 팀

> **이 노트북의 목표**
> 리서치·작성·검토 세 전문 에이전트를 만들고, `create_supervisor`로 감독자를 두어 "보고서를 조사부터 검토까지" 요청을 처리함. 감독자의 작업 분배와 핸드오프를 messages로 추적함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~14 · 멀티에이전트 = 사다리의 맨 위

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai langgraph langgraph-supervisor

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
# @tool 데코레이터: 일반 파이썬 함수를 LangChain "도구"로 변환
#   → 함수의 docstring이 도구 설명이 되고, 파라미터가 도구 입력 스키마가 됨
#   → 에이전트가 이 설명을 읽고 언제 이 도구를 쓸지 스스로 판단함
from langchain_core.tools import tool
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-3-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.3)
print("준비 완료")

## 1. 영역별 도구 준비

각 전문가가 쓸 도구를 영역별로 나눔. (실제론 검색·DB 등, 여기선 데모용)

In [ ]:
# 리서치 도구
@tool
def search_reference(query: str) -> str:
    """주제 관련 사실·수치를 조사한다. 자료 조사가 필요할 때 사용한다."""
    facts = {
        "생성형 AI": "2025년 기업 도입률 2배 증가, 반복업무 30% 시간 단축, 리스크는 환각·보안.",
    }
    for k, v in facts.items():
        if k in query:
            return v
    return "관련 자료: 일반적인 도입 효과와 리스크가 보고됨."

# 작성 도구
@tool
def write_report(topic: str, facts: str) -> str:
    """주제와 조사 자료로 보고서를 작성한다. 글 작성이 필요할 때 사용한다."""
    return f"[보고서] {topic}\n근거: {facts}\n→ 도입 효과가 크나 리스크 관리가 필요함."

# 검토 도구
@tool
def review_doc(text: str) -> str:
    """문서의 사실·완성도를 검토한다. 검토가 필요할 때 사용한다."""
    has_evidence = any(ch.isdigit() for ch in text)
    return "검토 통과: 근거 포함, 구조 양호." if has_evidence else "보강 필요: 근거 부족."

print("영역별 도구 준비 완료")

## 2. 전문 에이전트 3명 — 좁고 깊게

각자 자기 영역 도구·프롬프트만 가짐. 감독자가 참조하도록 `name` 부여.

In [ ]:
research_agent = create_agent(
    model=model, tools=[search_reference],
    system_prompt="너는 리서치 전문가야. 주제 관련 사실과 수치를 조사해 정리해.",
    name="researcher",
)

writer_agent = create_agent(
    model=model, tools=[write_report],
    system_prompt="너는 글쓰기 전문가야. 조사된 자료를 바탕으로 보고서를 작성해.",
    name="writer",
)

reviewer_agent = create_agent(
    model=model, tools=[review_doc],
    system_prompt="너는 검토 전문가야. 문서의 사실과 완성도를 점검해.",
    name="reviewer",
)

print("전문 에이전트 3명 생성: researcher, writer, reviewer")

## 3. 감독자(supervisor) — 작업 분배

`create_supervisor`로 감독자를 만듦. prompt에 "어떤 일은 누구에게"를 명확히.

In [ ]:
from langgraph_supervisor import create_supervisor

supervisor = create_supervisor(
    agents=[research_agent, writer_agent, reviewer_agent],
    model=model,
    prompt=(
        "너는 문서 제작 팀의 감독자야.\n"
        "- 자료 조사가 필요하면 researcher에게 넘겨.\n"
        "- 글 작성은 writer에게, 검토는 reviewer에게 넘겨.\n"
        "- 보통 조사 → 작성 → 검토 순으로 진행하고, 끝나면 사용자에게 종합해 답해."
    ),
# .compile(): 그래프를 실행 가능한 형태로 컴파일 (체크포인터 연결 등)
).compile()

print("감독자(supervisor) 완성 — 팀 조율 준비됨")

## 4. 실행 — 감독자가 팀을 조율한다

"조사부터 검토까지" 요청. 감독자가 각 전문가에게 분배하고, 핸드오프가 일어나는 과정을 추적함.

In [ ]:
# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
result = supervisor.invoke({
    "messages": [("user", "생성형 AI 도입 보고서를 자료 조사부터 작성·검토까지 해줘")]
})

# 누가 무슨 일을 했는지 흐름 추적
print("=== 팀 작업 흐름 ===")
for m in result["messages"]:
    role = type(m).__name__
    name = getattr(m, "name", None)
    if getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            who = name or "supervisor"
            print(f"[{who}] 🔧 {tc['name']}")
    elif role == "ToolMessage":
        print(f"   👁 결과: {m.content[:50]}")
print("\n=== 최종 종합 답변 ===")
print(result["messages"][-1].content)

> 📌 감독자가 researcher → writer → reviewer로 작업을 넘기고(핸드오프), 각 전문가의 결과가 공유 상태(messages)에 쌓이며 다음 전문가에게 전달됨.

## 5. 단일 에이전트와 비교 — 언제 멀티인가

같은 일을 단일 에이전트(모든 도구를 한 명에게)로도 할 수 있음. 차이를 생각해봄.

In [ ]:
# 단일 에이전트: 모든 도구를 한 명에게
solo = create_agent(
    model=model,
    tools=[search_reference, write_report, review_doc],
    system_prompt="너는 혼자 조사·작성·검토를 다 하는 만능 비서야.",
)
r = solo.invoke({"messages": [("user", "생성형 AI 도입 보고서를 조사·작성·검토해줘")]})
print("[단일 에이전트] 도구 3개 → 가능하지만, 도구가 더 늘면 선택이 흐려짐")
print(r["messages"][-1].content[:120], "...")

print("\n--- 판단 ---")
print("• 도구 3개, 단순 흐름 → 단일로 충분 (멀티는 과할 수 있음)")
print("• 도구가 10개+, 전문성이 뚜렷이 갈림 → 멀티가 유리")

> 📌 **남용 경계**: 멀티에이전트는 가장 무거움(감독자 판단·핸드오프·추가 호출). "단일로 안 되는가?"를 먼저 자문. 도구 과다·전문성 분리가 필요할 때만 올림(사다리의 맨 위).

## ✏️ 미니 실습

**과제**: 위 팀에 "번역 전문가" `translator_agent`를 추가하고(도구: `translate`), 감독자 prompt에 "번역이 필요하면 translator에게" 한 줄을 넣어, "이 보고서를 영어로 번역까지 해줘" 요청 처리하기.

In [ ]:
# TODO: 직접 작성
# @tool
# def translate(text: str, target: str) -> str:
#     """텍스트를 목표 언어로 번역한다."""
#     return f"[{target} 번역] ..."
# translator_agent = create_agent(model=model, tools=[translate],
#     system_prompt="너는 번역 전문가야.", name="translator")
# supervisor2 = create_supervisor(agents=[...4명...], model=model, prompt="...번역은 translator에게...").compile()

<details><summary>👉 정답</summary>

```python
@tool
def translate(text: str, target: str) -> str:
    """텍스트를 목표 언어로 번역한다. 번역이 필요할 때 사용한다."""
    return f"[{target} translation] (번역된 내용)"

translator_agent = create_agent(model=model, tools=[translate],
    system_prompt="너는 번역 전문가야. 문서를 요청한 언어로 번역해.", name="translator")

supervisor2 = create_supervisor(
    agents=[research_agent, writer_agent, reviewer_agent, translator_agent],
    model=model,
    prompt="너는 감독자야. 조사는 researcher, 작성은 writer, 검토는 reviewer, 번역은 translator에게 넘겨.",
).compile()

r = supervisor2.invoke({"messages": [("user", "생성형 AI 보고서를 작성하고 영어로 번역까지 해줘")]})
print(r["messages"][-1].content)
```
</details>

## 정리

| 절 | 한 일 | 핵심 |
|---|---|---|
| 1~2 | 전문 에이전트 3명 | 영역별 도구·프롬프트, name 부여 |
| 3 | create_supervisor | 감독자가 작업 분배 |
| 4 | 실행·추적 | 핸드오프 + 공유 상태 |
| 5 | 단일 vs 멀티 | 복잡성 비용 경계 |

### 3줄 요약
1. 전문 에이전트를 좁고 깊게 나누고, supervisor가 작업을 분배함.
2. 핸드오프로 제어·맥락을 넘기고 공유 상태로 결과가 쌓임.
3. 멀티는 가장 무거운 구조 — 도구 과다·전문성 분리 때만 올림.

### ▶ 다음 (Part 16)
만든 시스템이 잘 작동하는지 어떻게 알까? **관측·평가(LangSmith)** — 모든 단계를 추적하고 답의 품질을 측정함.